# **Project work: A mini segmentation challenge**

<div style="color:#777777;margin-top: -15px;">
<b>Course</b>: MSLS / CO4 |
<b>Version</b>: v1.3 <br><br>
<!-- 10.04.2025, v1.2: Fully refactored -->
<!-- 17.04.2026, v1.3: Introduced word count limits -->
</div>


**Student**: Oliver Hertach  
**Email**: hertaoli@students.zhaw.ch  
**University**: ZHAW  
**Semester**: 2. Semester, SS26   
**Date**: 31.05.2026


<br>

## **Abstract**

As Wolfgang Güllich famously noted, the brain is the most important muscle in climbing [1]. This reflects the fact that climbing involves a near infinite number of possible movement choices, making the optimal sequence of holds (“beta”) the key challenge.

Currently, beta is typically shared through videos, which are data-heavy and require moderation on platforms. An alternative representation could encode climbs as structured sequences of holds with spatial coordinates and types, rather than full video data. Such a representation would be significantly more compact and could enable automated reconstruction or visualization of climbing sequences.

As a first step toward this goal, this work focuses on the segmentation of climbing holds from wall images.

(This project is also on [GitHub](https://github.com/Oli36167/ClimbingVault/tree/main))

<br><br>

----

## **Table of contents**
<!-- Unfortunately, the following does not always work correctly -->
* [1. Dataset](#sec_dataset)  
* [2. Preprocessing](#sec_preprocessing)  
* [3. Manual segmentation](#sec_manual_segmentation)  
* [4. Automated segmentation](#sec_automated_segmentation)  
* [5. Evaluation and Discussion](#sec_evaluation)  
* [Appendix: Hints](#sec_hints)  


---

## **Prerequisites / Setup**

In [ ]:
import os
from copy import deepcopy
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

os.chdir(Path.cwd().parent)
print("new cwd:", Path.cwd())

---


<a id='sec_dataset'></a>

## **Dataset**

Using this dataset:

https://www.kaggle.com/datasets/tomasslama/indoor-climbing-gym-hold-segmentation/data?select=sm

10 pictures were chosen for the project. To not overcomplicate things, mostly pictures where only one wall at the time is visible, with clearly visible and not too many holds were chosen. The pictures were from the "sm" folder from the dataset, and the original numbering was kept (e.g. 008.jpg). 

---

<a id='sec_preprocessing'></a>

## **Preprocessing**

No explicit preprocessing step was applied. While performance could potentially be improved by first separating the climbing wall from the background, such an approach would either require manual segmentation, contradicting the goal of fully automated segmentation, or introduce additional complexity beyond the scope of this project.



---

<a id='sec_manual_segmentation'></a>

## **Manual segmentation**

The pictures were manually segmented using Fiji. For each climbing hold color, a binary mask was created and stored as .png, with white pixels representing the holds and black pixels the background.  

Below, the first two original pictures on the left and all manually added masks on the right are shown. It is already visible, that the white chalk on some of the holds has quite an influence on the color of the holds. 

### Constants

In [ ]:
COLORS = {
    "red": [255, 0, 0],
    "blue": [0, 0, 255],
    "green": [0, 255, 0],
    "yellow": [255, 255, 0],
    "orange": [255, 165, 0],
    "purple": [128, 0, 128],
    "pink": [255, 105, 180],
    "black": [0, 0, 0],
    "downclimb": [128, 128, 128],
}

# These values seemed to have quite an impact on the
# dice score and were "semi automatically "tuned using tune_hsv_color(),
# which is at the very end of the script.
THRESHOLDS = {
    "HSV": {
        "red": {"lower": [0, 80, 80], "upper": [180, 255, 255]},
        "green": {"lower": [35, 50, 50], "upper": [85, 255, 255]},
        "blue": {"lower": [90, 50, 50], "upper": [130, 255, 255]},
        "yellow": {"lower": [20, 80, 80], "upper": [35, 255, 255]},
        "orange": {"lower": [10, 80, 80], "upper": [20, 255, 255]},
        "purple": {"lower": [140, 50, 50], "upper": [160, 255, 255]},
        "pink": {"lower": [160, 50, 50], "upper": [180, 255, 255]},
        "black": {"lower": [60, 0, 0], "upper": [120, 255, 60]},
        "downclimb": {"lower": [0, 0, 60], "upper": [180, 50, 200]},
    },
    "RGB": {
        "red": {"lower": [150, 0, 0], "upper": [255, 100, 100]},
        "green": {"lower": [0, 150, 0], "upper": [100, 255, 100]},
        "blue": {"lower": [0, 0, 150], "upper": [100, 100, 255]},
        "yellow": {"lower": [150, 150, 0], "upper": [255, 255, 120]},
        "orange": {"lower": [150, 80, 0], "upper": [255, 180, 100]},
        "purple": {"lower": [120, 0, 120], "upper": [200, 100, 255]},
        "pink": {"lower": [180, 100, 150], "upper": [255, 180, 220]},
        "black": {"lower": [0, 0, 0], "upper": [60, 60, 60]},
        "downclimb": {"lower": [120, 120, 120], "upper": [200, 200, 200]},
    },
    "YUV": {
        "red": {"lower": [0, 0, 150], "upper": [255, 120, 255]},
        "green": {"lower": [0, 0, 0], "upper": [255, 120, 140]},
        "blue": {"lower": [0, 140, 0], "upper": [255, 255, 255]},
        "yellow": {"lower": [150, 0, 0], "upper": [255, 120, 120]},
        "orange": {"lower": [120, 0, 100], "upper": [255, 140, 200]},
        "purple": {"lower": [100, 80, 120], "upper": [200, 150, 255]},
        "pink": {"lower": [120, 80, 150], "upper": [255, 160, 255]},
        "black": {"lower": [0, 0, 0], "upper": [80, 80, 80]},
        "downclimb": {"lower": [60, 60, 60], "upper": [180, 180, 180]},
    },
}

COLORS_LIST = list(COLORS.keys())

METHODS = ["RGB", "YUV", "HSV"]

IMG_DIR = Path("data/images")

IMAGE_IDS = sorted([f.stem for f in IMG_DIR.glob("*.jpg")])
print(IMAGE_IDS)

In [ ]:
def load_sample(img_id):
    img = cv.imread(f"data/images/{img_id}.jpg")

    mask_dir = f"data/masks/{img_id}"

    masks = {}

    for file in os.listdir(mask_dir):
        if file.endswith(".png"):
            color = file.replace(".png", "")
            masks[color] = cv.imread(os.path.join(mask_dir, file), 0)

    return img, masks

In [ ]:
def dice_score(gt, pred):
    intersection = np.sum(gt * pred)
    return (2 * intersection) / (np.sum(gt) + np.sum(pred))

In [ ]:
def visualize(img, masks):
    masks_bin = {k: (v > 0).astype(np.uint8) for k, v in masks.items()}

    img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    overlay = img_rgb.copy()

    for name, mask in masks_bin.items():
        overlay[mask == 1] = COLORS[name]

    # side-by-side visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # original image
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # overlay image
    axes[1].imshow(overlay)
    axes[1].set_title("Manual Segmentation Overlay")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    all_holds = np.zeros_like(next(iter(masks_bin.values())))

    for m in masks_bin.values():
        all_holds = np.logical_or(all_holds, m)

    all_holds = all_holds.astype(np.uint8)

    return all_holds

In [ ]:
for img_id in IMAGE_IDS[:2]:
    img, masks = load_sample(img_id)
    visualize(img, masks)

---

<a id='sec_automated_segmentation'></a>

## **Automated Segmentation**

The automated segmentation pipeline was evaluated in several stages. First, RGB, HSV, and YUV color spaces were compared using multiple methods such as blurring and denoising, with the Dice score used as the evaluation metric.

Based on these results, HSV was selected as the best performing color space and further investigated using HSV equalization and CLAHE. These enhanced pipelines were compared against the baseline HSV segmentation without additional methods.

Finally, a qualitative evaluation was performed using a exemplary image and the green holds to visually compare baseline and improved segmentation results.


### Combined plotting of RGB vs HSV vs YUV

To gain intuition about the different color spaces, each channel of RGB, HSV, and YUV was visualized separately using a grayscale colormap. As a first baseline, segmentation masks were created for the first image using the color green. The results show that HSV provides a noticeably clearer separation of the green holds compared to RGB and YUV.

In [ ]:
IMG_ID = "008"

img, masks = load_sample(IMG_ID)

# Convert color spaces
img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
yuv = cv.cvtColor(img, cv.COLOR_BGR2YUV)

# Ground truth
gt = (masks["green"] > 0).astype(np.uint8)

# -----------------------------
# RGB segmentation
# -----------------------------
rgb_lower = np.array([0, 80, 0])
rgb_upper = np.array([180, 255, 180])

rgb_mask = cv.inRange(img_rgb, rgb_lower, rgb_upper)
rgb_pred = (rgb_mask > 0).astype(np.uint8)

# -----------------------------
# HSV segmentation
# -----------------------------
hsv_lower = np.array([35, 50, 50])
hsv_upper = np.array([85, 255, 255])

hsv_mask = cv.inRange(hsv, hsv_lower, hsv_upper)
hsv_pred = (hsv_mask > 0).astype(np.uint8)

# -----------------------------
# YUV segmentation
# -----------------------------
yuv_lower = np.array([0, 40, 0])
yuv_upper = np.array([255, 120, 140])

yuv_mask = cv.inRange(yuv, yuv_lower, yuv_upper)
yuv_pred = (yuv_mask > 0).astype(np.uint8)

# -----------------------------
# Dice scores
# -----------------------------
rgb_dice = dice_score(gt, rgb_pred)
hsv_dice = dice_score(gt, hsv_pred)
yuv_dice = dice_score(gt, yuv_pred)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

# -------------------
# RGB
# -------------------

# title
axes[0, 0].set_ylabel("RGB", rotation=0, labelpad=40)
axes[0, 1].set_ylabel("HSV", rotation=0, labelpad=40)
axes[0, 2].set_ylabel("YUV", rotation=0, labelpad=40)

axes[0, 0].imshow(img_rgb[:, :, 0], cmap="gray")
axes[0, 0].set_title("R")
axes[0, 0].axis("off")

axes[1, 0].imshow(img_rgb[:, :, 1], cmap="gray")
axes[1, 0].set_title("G")
axes[1, 0].axis("off")

axes[2, 0].imshow(img_rgb[:, :, 2], cmap="gray")
axes[2, 0].set_title("B")
axes[2, 0].axis("off")


# -------------------
# YUV
# -------------------
axes[0, 1].imshow(yuv[:, :, 0], cmap="gray")
axes[0, 1].set_title("Y")
axes[0, 1].axis("off")

axes[1, 1].imshow(yuv[:, :, 1], cmap="gray")
axes[1, 1].set_title("U")
axes[1, 1].axis("off")

axes[2, 1].imshow(yuv[:, :, 2], cmap="gray")
axes[2, 1].set_title("V")
axes[2, 1].axis("off")


# -------------------
# HSV
# -------------------
axes[0, 2].imshow(hsv[:, :, 0], cmap="gray")
axes[0, 2].set_title("H")
axes[0, 2].axis("off")

axes[1, 2].imshow(hsv[:, :, 1], cmap="gray")
axes[1, 2].set_title("S")
axes[1, 2].axis("off")

axes[2, 2].imshow(hsv[:, :, 2], cmap="gray")
axes[2, 2].set_title("V")
axes[2, 2].axis("off")

plt.tight_layout()
plt.show()


# -----------------------------
# MASK VISUALIZATION
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb_mask, cmap="gray")
axes[0].set_title(f"RGB Mask\nDice: {rgb_dice:.3f}")
axes[0].axis("off")


axes[1].imshow(yuv_mask, cmap="gray")
axes[1].set_title(f"YUV Mask\nDice: {yuv_dice:.3f}")
axes[1].axis("off")

axes[2].imshow(hsv_mask, cmap="gray")
axes[2].set_title(f"HSV Mask\nDice: {hsv_dice:.3f}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# OVERLAY VISUALIZATION
# -----------------------------
rgb_overlay = img_rgb.copy()
rgb_overlay[rgb_pred == 1] = [0, 255, 0]

hsv_overlay = img_rgb.copy()
hsv_overlay[hsv_pred == 1] = [0, 255, 0]

yuv_overlay = img_rgb.copy()
yuv_overlay[yuv_pred == 1] = [0, 255, 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb_overlay)
axes[0].set_title("RGB Overlay")
axes[0].axis("off")


axes[1].imshow(yuv_overlay)
axes[1].set_title("YUV Overlay")
axes[1].axis("off")

axes[2].imshow(hsv_overlay)
axes[2].set_title("HSV Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### Segmention functions

In [ ]:
def segment_color(img, color, method, THRESHOLDS):
    method = method.upper()

    if method == "HSV":
        img_cvt = cv.cvtColor(img, cv.COLOR_BGR2HSV)
    elif method == "RGB":
        img_cvt = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    elif method == "YUV":
        img_cvt = cv.cvtColor(img, cv.COLOR_BGR2YUV)
    else:
        raise ValueError(method)

    lower = np.array(THRESHOLDS[method][color]["lower"], dtype=np.uint8)
    upper = np.array(THRESHOLDS[method][color]["upper"], dtype=np.uint8)

    mask = cv.inRange(img_cvt, lower, upper)
    return (mask > 0).astype(np.uint8)

---
### Additional Methods

In [ ]:
# image
def blur(img):
    return cv.GaussianBlur(img, (5, 5), 0)


def equalize_hsv(img):
    hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
    hsv[:, :, 2] = cv.equalizeHist(hsv[:, :, 2])
    return cv.cvtColor(hsv, cv.COLOR_HSV2BGR)


def denoise(img):
    return cv.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)


def apply_pipeline(img, pipeline):
    for stage, func in pipeline:
        if stage == "image":
            img = func(img)
    return img

In [ ]:
# mask
def open_mask(mask):
    kernel = np.ones((5, 5), np.uint8)
    return cv.morphologyEx(mask.astype(np.uint8), cv.MORPH_OPEN, kernel)


def close_mask(mask):
    kernel = np.ones((3, 3), np.uint8)
    return cv.morphologyEx(mask.astype(np.uint8), cv.MORPH_CLOSE, kernel)


def connected_components_filter(mask, min_area=500):

    mask = mask.astype(np.uint8)

    if len(mask.shape) == 3:
        mask = cv.cvtColor(mask, cv.COLOR_BGR2GRAY)

    num_labels, labels, stats, _ = cv.connectedComponentsWithStats(mask, connectivity=8)

    cleaned = np.zeros_like(mask)

    for i in range(1, num_labels):  # skip background
        area = stats[i, cv.CC_STAT_AREA]

        if area >= min_area:
            cleaned[labels == i] = 1

    return cleaned.astype(np.uint8)


def apply_mask_pipeline(mask, pipeline):
    for stage, step in pipeline:
        if stage == "mask":
            mask = step(mask)

    return mask

In [ ]:
from pathlib import Path


def run_experiment(
    IMAGE_IDS,
    METHODS,
    COLORS,
    THRESHOLDS,
    pipeline=None,
    store=False,
):

    results = []

    for img_id in IMAGE_IDS:
        img, masks = load_sample(img_id)

        # -------------------------
        # IMAGE PIPELINE (ONCE)
        # -------------------------
        img_proc = img.copy()

        if pipeline:
            for stage, func in pipeline:
                if stage == "image":
                    img_proc = func(img_proc)

        # -------------------------
        # LOOP COLORS / METHODS
        # -------------------------
        for color in COLORS:
            if color not in masks:
                continue

            gt = (masks[color] > 0).astype(np.uint8)

            for method in METHODS:
                pred = segment_color(img_proc, color, method, THRESHOLDS)

                # -------------------------
                # MASK PIPELINE
                # -------------------------
                if pipeline:
                    for stage, func in pipeline:
                        if stage == "mask":
                            pred = func(pred)

                # -------------------------
                # OPTIONAL SAVE
                # -------------------------
                if store:
                    out_dir = Path("data") / "masks" / "automated" / img_id / color

                    out_dir.mkdir(parents=True, exist_ok=True)

                    cv.imwrite(
                        str(out_dir / f"{method}.png"), pred.astype(np.uint8) * 255
                    )

                score = dice_score(gt, pred)

                results.append(
                    {
                        "image": img_id,
                        "color": color,
                        "method": method,
                        "dice": score,
                    }
                )

    return pd.DataFrame(results)


# def run_experiment(IMAGE_IDS, METHODS, COLORS, THRESHOLDS, pipeline=None):

#     results = []

#     for img_id in IMAGE_IDS:
#         img, masks = load_sample(img_id)

#         # -------------------------
#         # IMAGE PIPELINE (ONCE)
#         # -------------------------
#         img_proc = img.copy()

#         if pipeline:
#             for stage, func in pipeline:
#                 if stage == "image":
#                     img_proc = func(img_proc)

#         # -------------------------
#         # LOOP COLORS / METHODS
#         # -------------------------
#         for color in COLORS:
#             if color not in masks:
#                 continue

#             gt = (masks[color] > 0).astype(np.uint8)

#             for method in METHODS:
#                 pred = segment_color(img_proc, color, method, THRESHOLDS)

#                 # -------------------------
#                 # MASK PIPELINE
#                 # -------------------------
#                 if pipeline:
#                     for stage, func in pipeline:
#                         if stage == "mask":
#                             pred = func(pred)

#                 score = dice_score(gt, pred)

#                 results.append(
#                     {"image": img_id, "color": color, "method": method, "dice": score}
#                 )

#     return pd.DataFrame(results)

---
### Running and Evaluating of Baseline

In [ ]:
def compute_heatmap(df, METHODS):
    return df.pivot_table(
        index="color", columns="method", values="dice", aggfunc="mean"
    ).reindex(columns=METHODS)


def evaluate(df, METHODS, title="Evaluation"):

    pivot = compute_heatmap(df, METHODS)

    # -------------------------
    # GLOBAL METHOD SCORES
    # -------------------------
    method_means = df.groupby("method")["dice"].mean().reindex(METHODS)

    # -------------------------
    # HEATMAP
    # -------------------------
    fig, ax = plt.subplots(figsize=(8, 5))

    im = ax.imshow(pivot, aspect="auto", vmin=0, vmax=1)

    # -------------------------
    # COLUMN LABELS + MEANS
    # -------------------------
    labels = [f"{method}\n{method_means[method]:.2f}" for method in pivot.columns]

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(labels)

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    # -------------------------
    # CELL ANNOTATIONS
    # -------------------------
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = pivot.iloc[i, j]

            if pd.notna(value):
                ax.text(
                    j,
                    i,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    color="white" if value < 0.5 else "black",
                )

    fig.colorbar(im, ax=ax, label="Dice score")

    ax.set_title(title)

    plt.tight_layout()
    plt.show()

---
### Baseline Heatmap

The heatmap below confirms the earlier observation from the overlay visualization for green holds: HSV consistently outperforms RGB and YUV, achieving a Dice score of 0.27. The only exception are blue holds that YUV seems to segment better. 

The results also reveal strong variability across different colors. While pink holds are segmented relatively well, the segmentation for grey downclimb holds and purple holds fails.

In the next step, different image processing techniques such as denoising and blurring are evaluated to improve segmentation performance.


In [ ]:
methods = ["RGB", "YUV", "HSV"]

# base
df_base = run_experiment(IMAGE_IDS, methods, COLORS_LIST, THRESHOLDS)


evaluate(df_base, methods, "Baseline")


# blur
df_blur = run_experiment(
    IMAGE_IDS, methods, COLORS_LIST, THRESHOLDS, pipeline=[("image", blur)]
)

# close_mask
df_close = run_experiment(
    IMAGE_IDS, methods, COLORS_LIST, THRESHOLDS, pipeline=[("mask", close_mask)]
)

# open_mask
df_open = run_experiment(
    IMAGE_IDS, methods, COLORS_LIST, THRESHOLDS, pipeline=[("mask", open_mask)]
)


df_cc = run_experiment(
    IMAGE_IDS,
    methods,
    COLORS_LIST,
    THRESHOLDS,
    pipeline=[("mask", connected_components_filter)],
)

# denoise
df_denoise = run_experiment(
    IMAGE_IDS, methods, COLORS_LIST, THRESHOLDS, pipeline=[("image", denoise)]
)

In [ ]:
def summarize(df, METHODS):
    return df.groupby("method")["dice"].mean().reindex(METHODS)


def build_comparison_table(pipelines, METHODS):
    data = {}

    for name, config in pipelines.items():
        df = config["df"]

        data[name] = df.groupby("method")["dice"].mean().reindex(METHODS)

    return pd.DataFrame(data)

In [ ]:
def plot_pipeline_comparison(table, pipelines):

    x = np.arange(len(table.index))
    width = 0.8 / len(table.columns)

    plt.figure(figsize=(10, 5))

    baseline_col = table.columns[0]
    baseline = table[baseline_col]

    # bars
    for i, col in enumerate(table.columns):
        values = table[col].values
        color = pipelines[col]["color"]

        for j, value in enumerate(values):
            worse_than_baseline = value < baseline.iloc[j]

            plt.bar(
                x[j] + i * width,
                value,
                width,
                color=color,
                hatch="//" if worse_than_baseline else None,
                alpha=0.9,
            )

    # clean custom legend
    legend_handles = [
        Patch(facecolor=config["color"], label=name)
        for name, config in pipelines.items()
    ]

    plt.legend(handles=legend_handles)

    plt.xticks(x + width * (len(table.columns) - 1) / 2, table.index)

    plt.ylabel("Mean Dice Score")
    plt.title("Pipeline Comparison")
    plt.ylim(0, 1)

    plt.show()

In [ ]:
pipelines = {
    "baseline": {
        "df": df_base,
        "color": "steelblue",
    },
    "blur": {
        "df": df_blur,
        "color": "orange",
    },
    "denoise": {
        "df": df_denoise,
        "color": "green",
    },
    "open": {
        "df": df_open,
        "color": "purple",
    },
    "close_mask": {
        "df": df_close,
        "color": "red",
    },
    "connected_components": {
        "df": df_cc,
        "color": "brown",
    },
}

---
### Methods

The following image processing methods, blur, denoising, opening, closing, and connected components filtering, were applied across the RGB, YUV, and HSV color spaces.

The results confirm earlier observations: HSV consistently outperforms RGB and YUV, even when additional preprocessing methods are introduced. Methods that decreased the dice score relative to the baseline are marked with lines in the evaluation plots.

For subsequent experiments, only the HSV color space and those preprocessing methods that improve performance are considered.

In [ ]:
table = build_comparison_table(pipelines, METHODS)
plot_pipeline_comparison(table, pipelines)

---
### HSV specific optimizing

In [ ]:
def clahe_hsv(img, clip_limit=2.0, tile_grid_size=(8, 8)):

    hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)

    h, s, v = cv.split(hsv)

    clahe = cv.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    v_clahe = clahe.apply(v)

    hsv_clahe = cv.merge([h, s, v_clahe])

    return cv.cvtColor(hsv_clahe, cv.COLOR_HSV2BGR)

In [ ]:
base_hsv_pipeline = [
    ("image", blur),
    ("image", denoise),
    # ("mask", open_mask), # excluded as this seems to lower the dice score
    ("mask", close_mask),
    ("mask", connected_components_filter),
]

In [ ]:
pipeline_eq = base_hsv_pipeline + [("image", equalize_hsv)]
pipeline_clahe = base_hsv_pipeline + [("image", clahe_hsv)]
pipeline_both = base_hsv_pipeline + [("image", equalize_hsv)] + [("image", clahe_hsv)]

df_hsv_base = run_experiment(IMAGE_IDS, ["HSV"], COLORS_LIST, THRESHOLDS, pipeline=[])

df_hsv_full = run_experiment(
    IMAGE_IDS, ["HSV"], COLORS_LIST, THRESHOLDS, pipeline=base_hsv_pipeline
)

df_hsv_eq = run_experiment(
    IMAGE_IDS, ["HSV"], COLORS_LIST, THRESHOLDS, pipeline=pipeline_eq, store=True
)

df_hsv_clahe = run_experiment(
    IMAGE_IDS, ["HSV"], COLORS_LIST, THRESHOLDS, pipeline=pipeline_clahe
)

df_hsv_both = run_experiment(
    IMAGE_IDS, ["HSV"], COLORS_LIST, THRESHOLDS, pipeline=pipeline_both
)

In [ ]:
pipelines_hsv = {
    "baseline HSV": {"df": df_hsv_base, "color": "steelblue"},
    "full pipeline": {"df": df_hsv_full, "color": "orange"},
    "+ equalize": {"df": df_hsv_eq, "color": "green"},
    "+ CLAHE": {"df": df_hsv_clahe, "color": "purple"},
    "+ both": {"df": df_hsv_both, "color": "red"},
}

---
### Pipeline comparison

The methods blur, denoising, morphological closing, and connected components filtering were combined into a full pipeline. This pipeline was further extended using either HSV equalization or CLAHE.

Among the tested configurations, the full pipeline combined with HSV equalization achieves the highest overall Dice score.

In [ ]:
table_hsv = build_comparison_table(pipelines_hsv, ["HSV"])
plot_pipeline_comparison(table_hsv, pipelines_hsv)

In [ ]:
def compare_heatmaps(experiments, METHODS):

    n = len(experiments)

    fig, axes = plt.subplots(1, n, figsize=(3 * n + 2, 5), sharey=True)

    if n == 1:
        axes = [axes]

    last_im = None

    for i, (ax, (name, df)) in enumerate(zip(axes, experiments.items())):
        pivot = compute_heatmap(df, METHODS)

        im = ax.imshow(pivot, aspect="auto", vmin=0, vmax=1)
        last_im = im

        # -------------------------
        # X labels
        # -------------------------
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)

        # -------------------------
        # Y labels (only once)
        # -------------------------
        ax.set_yticks(range(len(pivot.index)))

        if i == 0:
            ax.set_yticklabels(pivot.index)
        else:
            ax.tick_params(labelleft=False)

        # -------------------------
        # Title
        # -------------------------
        ax.set_title(name)

        # -------------------------
        # CELL ANNOTATIONS
        # -------------------------
        for y in range(pivot.shape[0]):
            for x in range(pivot.shape[1]):
                value = pivot.iloc[y, x]

                if pd.notna(value):
                    ax.text(
                        x,
                        y,
                        f"{value:.2f}",
                        ha="center",
                        va="center",
                        color="white" if value < 0.5 else "black",
                        fontsize=9,
                    )

        # -------------------------
        # MEAN BELOW EACH COLUMN
        # -------------------------
        col_means = df.groupby("method")["dice"].mean().reindex(pivot.columns)
        col_stds = df.groupby("method")["dice"].std().reindex(pivot.columns)

        for x, method in enumerate(pivot.columns):
            ax.text(
                x,
                len(pivot.index) + 0.3,
                f"{col_means[method]:.2f}\n±{col_stds[method]:.2f}",
                ha="center",
                va="top",
                fontsize=9,
                fontweight="bold",
            )

        # extend y-limit to make room for mean row
        ax.set_ylim(len(pivot.index) - 0.5, -0.8)

    # -------------------------
    # LAYOUT
    # -------------------------
    fig.subplots_adjust(wspace=0.05)

    cbar = fig.colorbar(last_im, ax=axes, fraction=0.03, pad=0.02)
    cbar.set_label("Dice score")

    plt.show()

---

<a id='sec_evaluation'></a>

## **Evaluation and Discussion**

The HSV baseline is now compared to the best-performing pipeline on a color basis. The overall mean dice score improves from 0.27 to 0.37. While the absolute increase is small, it still is a meaningful relative improvement.

A notable observation is that the model continues to struggle with certain classes, particularly purple holds and grey downclimb holds, where the dice score remains close to zero. Hovewer this is not surprising when looking at the dataset as large parts of the walls are also purple, and as downclimb holds often intentionally match the color of the walls in climbing gyms.

At the same time, the color blue show that alternative color spaces (YUV, 0.48 for blue) can occasionally outperform the optimized HSV pipeline, suggesting that HSV is not universally optimal across all colors.

One possible limitation of the current approach lies in the manually defined thresholds. These could be further improved through systematic tuning or optimization per class. An alternative direction would be an interactive threshold selection approach, where the user selects a representative pixel and thresholds are estimated from the local color distribution. However, this would shift the method from fully automatic segmentation toward a semi-automatic workflow.

In [ ]:
compare_heatmaps({"baseline": df_base, "best pipeline": df_hsv_eq}, ["HSV"])

---
Finally, the HSV baseline is compared visually using image 008.jpg with the green holds. It is clearly visible in both the segmentation masks and the Dice score that only parts of the green holds are correctly identified.

The Dice score improves only marginally from 0.51 in the baseline to 0.52 with the full pipeline. However, the structure of the predicted masks changes noticeably: some thin structures are lost, while small holes within detected regions are filled.

These effects are likely caused by the morphological operations in the pipeline, in particular morphological closing which fills gaps and holes and connected components filtering which removes small or fragmented regions.

Overall, further tuning and experimentation would be required to achieve robust performance across all hold colors. While some classes such as pink already achieve very high dice scores (0.96 across all pictures!), other classes remain challenging and require improved thresholding or further segmentation strategies.


In [ ]:
def make_overlay(img_rgb, mask, color=(0, 255, 0)):
    overlay = img_rgb.copy()
    overlay[mask == 1] = color
    return overlay


img, masks = load_sample(IMG_ID)
gt = (masks["green"] > 0).astype(np.uint8)

# -------------------------
# PIPELINES
# -------------------------
img_base = apply_pipeline(img, [])
pred_base = segment_color(img_base, "green", "HSV", THRESHOLDS)

img_full = apply_pipeline(img, pipeline_eq)
pred_full = segment_color(img_full, "green", "HSV", THRESHOLDS)
pred_full = apply_mask_pipeline(pred_full, pipeline_eq)

# -------------------------
# VISUAL BASE IMAGE
# -------------------------
img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)

overlay_gt = make_overlay(img_rgb, gt, color=(0, 255, 0))
overlay_base = make_overlay(img_rgb, pred_base, color=(0, 255, 0))
overlay_full = make_overlay(img_rgb, pred_full, color=(0, 255, 0))

# -------------------------
# PLOT 2x2
# -------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# 1.1 original image
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Original image")
axes[0, 0].axis("off")

# 1.2 ground truth
axes[0, 1].imshow(overlay_gt)
axes[0, 1].set_title("Ground truth (green holds)")
axes[0, 1].axis("off")

# 2.1 baseline
axes[1, 0].imshow(overlay_base)
axes[1, 0].set_title("HSV baseline")
axes[1, 0].axis("off")

# 2.2 best pipeline
axes[1, 1].imshow(overlay_full)
axes[1, 1].set_title("HSV + best pipeline")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def tune_hsv_color(
    IMAGE_IDS, color, THRESHOLDS, pipeline, lower_h_range, upper_h_range
):

    best_score = -1
    best_threshold = None

    for lower_h in lower_h_range:
        for upper_h in upper_h_range:
            if lower_h >= upper_h:
                continue

            # copy THRESHOLDS safely
            trial = deepcopy(THRESHOLDS)

            # modify ONLY this color
            trial["HSV"][color]["lower"][0] = lower_h
            trial["HSV"][color]["upper"][0] = upper_h

            # run experiment ONLY for HSV + one color
            df = run_experiment(
                image_ids=IMAGE_IDS,
                METHODS=["HSV"],
                colors=[color],
                THRESHOLDS=trial,
                pipeline=pipeline,
            )

            score = df["dice"].mean()

            if score > best_score:
                best_score = score
                best_threshold = {
                    "lower": trial["HSV"][color]["lower"].copy(),
                    "upper": trial["HSV"][color]["upper"].copy(),
                }

    return best_threshold, best_score

---
### Threshold Tuning
The following code can be used for semi-automatic threshold tuning. 

In [ ]:
# best_threshold, best_score = tune_hsv_color(
#     image_ids=IMAGE_IDS,
#     color="red",
#     thresholds=THRESHOLDS,
#     pipeline=pipeline_eq,
#     lower_h_range=range(0, 31, 5),
#     upper_h_range=range(150, 181, 5)
# )

# print("red")
# print(best_threshold)
# print(best_score)

---

<a id='sec_references'></a>

## **References**

[1] T. Hepp, Wolfgang Güllich - Leben in der Senkrechten: eine Biographie, Rosenheimer-Verl, Rosenheim, 1993.

[2] ChatGPT, de-DE, 2026, https://chatgpt.com/de-DE/.

[3] Claude, en-US, https://claude.ai/new.

[4] A. Adel, N. Alani, AI & SOCIETY 2025, DOI 10.1007/s00146-025-02406-7

## Usage of Large Language Models 

This report used two large language models as aids: ChatGPT and Claude [2, 3]. In the interest of time, they were used as an aid to write the code. They were also used to help with formulation and spelling.  

To make the literature search more efficient, ChatGPT was used as a source of inspiration to quickly identify the most relevant papers, specifically by asking it to provide key references on a given topic. However, ChatGPT is known to hallucinate [4]. Therefore, no content was taken directly from its responses about literature search and the suggested references and links were only used after their validity had been carefully verified. 
